# AI-Generated Text Detection: best TF-IDF run, focused on en-EN blogs
Competition: [ai-generated-text-detection-hw3-ucu-ai-2026](https://www.kaggle.com/competitions/ai-generated-text-detection-hw3-ucu-ai-2026), 3 classes: `human`, `ai`, `fusion`.

Same model as `tfidf_best_hf`, the best configuration from the `tfidf_classical` search:

| | |
|---|---|
| features | `hf_subword`: TF-IDF over 1–2-grams of `FacebookAI/xlm-roberta-base` subword ids (open-weight tokenizer from Hugging Face) |
| model | `LogisticRegression(C=16)` |
| search score | macro-F1 = 0.8594 (30k-row search split) |

**Focus: en-EN `blog_authorship_corpus`.** This is the largest English domain in test (4,446 rows) and has balanced classes. Its `fusion` texts are human blog posts with usually just **1–2 AI sentences inserted**, which makes `fusion` vs `human` hard. Three changes target it:
- **All blog rows** from the training pool go into the training set (`INCLUDE_ALL_FOCUS`).
- Blog rows get a higher sample weight (`FOCUS_WEIGHT`).
- Blog `fusion` rows, the hardest class there, get an extra multiplier (`FOCUS_FUSION_WEIGHT`).

The Gutenberg weighting from `tfidf_best_hf` stays on (set `GUT_WEIGHT = GUT_MINORITY_WEIGHT = 1` to switch it off). Weights are rescaled to mean 1, so `C` keeps its meaning.

**Validation caveat (blog twins)**: many blog `fusion` texts have their human original in train, with the same opening. A random split can put one twin in training and the other in validation, which flatters the score. Test texts have no such twins. The blog deep-dive therefore reports validation separately for rows **with** and **without** a twin in the training set; the no-twin number is the better estimate for test.

The notebook shows where the model fails (overall, per language × domain, Gutenberg, a blog deep-dive, by length, error examples) and ends by saving `submission.csv`.

**Running on Kaggle**: add the competition data, set Accelerator to **None** (CPU), and turn **Internet on** so the tokenizer can download (or attach the `xlm-roberta-base` tokenizer files as an input). Roughly 30–60 min.

## Setup & config

In [ ]:
import os, re, gc, time, socket, warnings
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, FileLink
from joblib import Parallel, delayed

from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (f1_score, accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

# ---------------- config ----------------
RANDOM_STATE        = 42
VAL_N               = 20_000     # identical validation split to the other notebooks
FINAL_N             = 150_000    # stratified training rows (None = all remaining rows; more RAM/time)
INCLUDE_ALL_GUT     = True       # add every Gutenberg row from the pool to the training set
GUT_WEIGHT          = 2.0        # sample weight for all Gutenberg rows
GUT_MINORITY_WEIGHT = 4.0        # extra multiplier for Gutenberg human/fusion (total 8x vs a normal row)
FOCUS_LANG          = "en-EN"
FOCUS_DOMAIN        = "blog_authorship_corpus"
INCLUDE_ALL_FOCUS   = True       # add every en-EN blog row from the pool to the training set
FOCUS_WEIGHT        = 2.0        # sample weight for en-EN blog rows
FOCUS_FUSION_WEIGHT = 1.5        # extra multiplier for en-EN blog fusion rows (total 3x)
PREFIX_CHARS        = 100        # texts sharing their first 100 chars count as twins
C                   = 16         # best LogReg C from the search
MAX_ITER            = 1000
HF_TOKENIZER        = "FacebookAI/xlm-roberta-base"
N_JOBS              = os.cpu_count() or 2

TEXT, LANG, DOMAIN, LABEL = "text", "lang", "domain", "label"
CLASSES = ["human", "ai", "fusion"]
LANGS = ["en-EN", "de-DE"]
GUT = "gutenberg"

@contextmanager
def timer(msg):
    t0 = time.time()
    print(f"[start] {msg} ...", flush=True)
    yield
    print(f"[done]  {msg}: {time.time() - t0:.1f}s", flush=True)

## Load data

In [ ]:
COMP = "ai-generated-text-detection-hw3-ucu-ai-2026"

def find_data_dir():
    for root in [Path("/kaggle/input"), Path("data"), Path("../data")]:
        if root.exists():
            hits = sorted(root.rglob("train.csv"))
            hits = [h for h in hits if COMP in str(h)] or hits
            if hits:
                return hits[0].parent
    raise FileNotFoundError("train.csv not found: add the competition data as an input")

DATA_DIR = find_data_dir()
with timer("read csv"):
    train = pd.read_csv(DATA_DIR / "train.csv")
    test = pd.read_csv(DATA_DIR / "test.csv")
    sample_sub = pd.read_csv(next(iter(sorted(DATA_DIR.glob("sample_submission*.csv")))))
for df in (train, test):
    df[TEXT] = df[TEXT].fillna("").astype(str)
print(f"data dir: {DATA_DIR} | train {train.shape} | test {test.shape}")

## Splits and weighted training set (en-EN blog + Gutenberg)

In [ ]:
strat = train[LABEL] + "|" + train[LANG] + "|" + train[DOMAIN]
pool_idx, val_idx = train_test_split(train.index, test_size=VAL_N, stratify=strat, random_state=RANDOM_STATE)
pool, val = train.loc[pool_idx], train.loc[val_idx]

def strat_sample(df, n, seed=RANDOM_STATE):
    if n is None or n >= len(df):
        return df
    s = df[LABEL] + "|" + df[LANG] + "|" + df[DOMAIN]
    keep, _ = train_test_split(df.index, train_size=n, stratify=s, random_state=seed)
    return df.loc[keep]

final_tr = strat_sample(pool, FINAL_N)
def focus_mask(df):
    return ((df[LANG] == FOCUS_LANG) & (df[DOMAIN] == FOCUS_DOMAIN)).values

for flag, mask_fn, name in [(INCLUDE_ALL_GUT, lambda d: (d[DOMAIN] == GUT).values, "Gutenberg"),
                            (INCLUDE_ALL_FOCUS, focus_mask, "en-EN blog")]:
    if flag:
        extra = pool[mask_fn(pool) & ~pool.index.isin(final_tr.index)]
        final_tr = pd.concat([final_tr, extra])
        print(f"added {len(extra):,} extra {name} rows")
del pool; gc.collect()

is_gut = (final_tr[DOMAIN] == GUT).values
is_focus = focus_mask(final_tr)
lab = final_tr[LABEL].values
minority = np.isin(lab, ["human", "fusion"])
weights = (np.where(is_gut, GUT_WEIGHT, 1.0) * np.where(is_gut & minority, GUT_MINORITY_WEIGHT, 1.0)
           * np.where(is_focus, FOCUS_WEIGHT, 1.0) * np.where(is_focus & (lab == "fusion"), FOCUS_FUSION_WEIGHT, 1.0))
weights = weights / weights.mean()          # mean 1 -> C keeps its meaning

# twins: validation rows whose opening also appears in the training set
train_prefixes = set(final_tr[TEXT].str[:PREFIX_CHARS])

print(f"train rows: {len(final_tr):,} | val rows: {len(val):,}")
summary = (final_tr.assign(w=weights, domain_group=np.select([is_gut, is_focus], ["gutenberg", "en-EN blog"], "other"))
           .groupby(["domain_group", LABEL]).agg(rows=("w", "size"), weight_per_row=("w", "mean"), total_weight=("w", "sum")))
display(summary.round(2))

## HF subword tokenizer (open-weight, from Hugging Face)
Loads `xlm-roberta-base`'s SentencePiece tokenizer, either from a Kaggle input that contains the tokenizer files or from the Hub. Only the tokenizer is used, not the model weights.

In [ ]:
def internet_ok(host="huggingface.co", port=443, timeout=3):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

def find_local_tokenizer():
    root = Path("/kaggle/input")
    if root.exists():
        for f in sorted(root.rglob("tokenizer.json")) + sorted(root.rglob("sentencepiece.bpe.model")):
            if "xlm-roberta" in str(f).lower() or "xlm_roberta" in str(f).lower():
                return str(f.parent)
    return None

from transformers import AutoTokenizer
TOK_SRC = find_local_tokenizer()
if TOK_SRC is None:
    if not internet_ok():
        raise RuntimeError("This notebook needs the xlm-roberta-base tokenizer: turn on Settings -> Internet "
                           "or attach the tokenizer files as an input.")
    TOK_SRC = HF_TOKENIZER
HF_TOK = AutoTokenizer.from_pretrained(TOK_SRC)
HF_TOK.model_max_length = 10**9
print(f"tokenizer from {TOK_SRC} | vocab={HF_TOK.vocab_size:,} | example:",
      HF_TOK.tokenize("Die Bundesregierung muss handeln, delve into it")[:14])

## Features: TF-IDF over subword-id 1–2-grams
Each text becomes its sequence of token ids. The ids are hashed into 2^20 columns (unigrams + bigrams), then TF-IDF with sublinear term frequency and L2 normalisation.

In [ ]:
HASHER = HashingVectorizer(analyzer="word", ngram_range=(1, 2), lowercase=False, token_pattern=r"\S+",
                           n_features=2**20, alternate_sign=False, norm=None, dtype=np.float32)

def token_strings(texts, batch=2000):
    texts = list(texts)
    out = []
    for i in range(0, len(texts), batch):
        ids = HF_TOK(texts[i:i + batch], add_special_tokens=False, verbose=False)["input_ids"]
        out.extend(" ".join(map(str, x)) for x in ids)
    return out

def hashed(texts, chunk=4000):
    toks = token_strings(texts)
    parts = Parallel(n_jobs=N_JOBS)(delayed(HASHER.transform)(toks[i:i + chunk]) for i in range(0, len(toks), chunk))
    return sp.vstack(parts).tocsr()

tfidf = TfidfTransformer(sublinear_tf=True)
with timer(f"features train ({len(final_tr):,} rows)"):
    X_tr = tfidf.fit_transform(hashed(final_tr[TEXT])).astype(np.float32).tocsr()
with timer("features val + test"):
    X_val = tfidf.transform(hashed(val[TEXT])).astype(np.float32).tocsr()
    X_test = tfidf.transform(hashed(test[TEXT])).astype(np.float32).tocsr()
print(f"X_tr {X_tr.shape}, nnz/doc={X_tr.nnz / X_tr.shape[0]:.0f}")

## Fit the model: LogisticRegression(C=16) with Gutenberg sample weights

In [ ]:
with timer(f"fit LogisticRegression(C={C})"):
    model = LogisticRegression(C=C, max_iter=MAX_ITER)
    model.fit(X_tr, final_tr[LABEL].values, sample_weight=weights)
print("iterations:", model.n_iter_)
del X_tr; gc.collect()

val_proba = model.predict_proba(X_val)
val_pred = model.classes_[val_proba.argmax(1)]
y_val = val[LABEL].values
v = val.assign(pred=val_pred, correct=val_pred == y_val, n_chars=val[TEXT].str.len(),
               has_twin_in_train=val[TEXT].str[:PREFIX_CHARS].isin(train_prefixes).values)

## Overall per-class metrics

In [ ]:
VAL_MACRO_F1 = f1_score(y_val, val_pred, average="macro")
print(f"Validation ({len(val):,} rows) | macro-F1 = {VAL_MACRO_F1:.4f} | accuracy = {accuracy_score(y_val, val_pred):.4f}\n")
print(classification_report(y_val, val_pred, labels=CLASSES, digits=4))

p, r, f, s = precision_recall_fscore_support(y_val, val_pred, labels=CLASSES)
ax = pd.DataFrame({"precision": p, "recall": r, "f1": f}, index=CLASSES).plot.bar(rot=0, figsize=(8, 4), ylim=(0, 1),
                                                                                  title="Per-class metrics (validation)")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=8)
plt.tight_layout(); plt.show()

## Confusion matrices

In [ ]:
def cm_plot(y_true, y_pred, title, ax, normalize=True):
    cm = confusion_matrix(y_true, y_pred, labels=CLASSES)
    data = cm / np.maximum(cm.sum(1, keepdims=True), 1) if normalize else cm
    sns.heatmap(data, annot=True, fmt=".3f" if normalize else "d", cmap="Blues", vmin=0, vmax=1 if normalize else None,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax)
    ax.set_title(title); ax.set_xlabel("predicted"); ax.set_ylabel("true")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_plot(y_val, val_pred, "Counts", axes[0], normalize=False)
cm_plot(y_val, val_pred, "Row-normalised (recall per true class)", axes[1])
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, len(LANGS), figsize=(13, 5))
for ax, lang in zip(axes, LANGS):
    m = (v[LANG] == lang).values
    cm_plot(y_val[m], val_pred[m], f"{lang} (n={m.sum():,}, macro-F1={f1_score(y_val[m], val_pred[m], average='macro'):.3f})", ax)
plt.tight_layout(); plt.show()

## Where does it fail? Language × domain

In [ ]:
rows = []
for (lang, dom), gr in v.groupby([LANG, DOMAIN]):
    f1s = f1_score(gr[LABEL], gr["pred"], labels=CLASSES, average=None, zero_division=0)
    rows.append({"lang": lang, "domain": dom, "n": len(gr),
                 **{f"n_{c}": int((gr[LABEL] == c).sum()) for c in CLASSES},
                 "macro_f1": f1_score(gr[LABEL], gr["pred"], average="macro", zero_division=0),
                 "accuracy": gr["correct"].mean(), **{f"f1_{c}": x for c, x in zip(CLASSES, f1s)}})
by_dom = pd.DataFrame(rows).sort_values("macro_f1")
display(by_dom.style.background_gradient(subset=["macro_f1", "f1_human", "f1_ai", "f1_fusion"], cmap="RdYlGn", vmin=0.3, vmax=1)
        .format(precision=3))

plt.figure(figsize=(9, 0.45 * len(by_dom) + 1))
sns.heatmap(by_dom.set_index(["lang", "domain"])[["f1_human", "f1_ai", "f1_fusion", "macro_f1"]],
            annot=True, fmt=".2f", cmap="RdYlGn", vmin=0.3, vmax=1)
plt.title("F1 per class, by language x domain (validation)"); plt.tight_layout(); plt.show()

## Gutenberg deep-dive
How the weighted model does on Gutenberg. With only 18 human and 28 fusion validation rows here, a single text moves F1 by several points: read these numbers as a direction, not a precise score.

In [ ]:
gm = (v[DOMAIN] == GUT).values
print(f"Gutenberg validation rows: {gm.sum()} | labels: {v.loc[gm, LABEL].value_counts().to_dict()}")
print(f"Gutenberg macro-F1 = {f1_score(y_val[gm], val_pred[gm], average='macro'):.4f}\n")
print(classification_report(y_val[gm], val_pred[gm], labels=CLASSES, digits=3, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_plot(y_val[gm], val_pred[gm], "Gutenberg: counts", axes[0], normalize=False)
cm_plot(y_val[gm], val_pred[gm], "Gutenberg: row-normalised", axes[1])
plt.tight_layout(); plt.show()

# how confident is the model on Gutenberg minority texts?
gp = pd.DataFrame(val_proba[gm], columns=model.classes_).assign(true=y_val[gm], pred=val_pred[gm])
plt.figure(figsize=(10, 4))
for i, c in enumerate(CLASSES):
    plt.subplot(1, 3, i + 1)
    sns.histplot(data=gp, x=c, hue="true", hue_order=CLASSES, bins=20, element="step", stat="density", common_norm=False)
    plt.title(f"P({c}) on Gutenberg val"); plt.xlim(0, 1)
plt.tight_layout(); plt.show()

## en-EN blog deep-dive
Per-class metrics and confusion matrices on the blog validation rows (~3k). Then the **twin split**: rows whose opening also appears in training vs rows without. Test has no twins, so the no-twin score is the better estimate for test.

In [ ]:
bm = focus_mask(v)
yb, pb = y_val[bm], val_pred[bm]
print(f"blog validation rows: {bm.sum():,} | labels: {v.loc[bm, LABEL].value_counts().to_dict()}")
print(f"blog macro-F1 = {f1_score(yb, pb, average='macro'):.4f}\n")
print(classification_report(yb, pb, labels=CLASSES, digits=4, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm_plot(yb, pb, "en-EN blog: counts", axes[0], normalize=False)
cm_plot(yb, pb, "en-EN blog: row-normalised", axes[1])
plt.tight_layout(); plt.show()

vb = v[bm]
tw = []
for flag, gr in vb.groupby("has_twin_in_train"):
    f1s = f1_score(gr[LABEL], gr["pred"], labels=CLASSES, average=None, zero_division=0)
    tw.append({"has_twin_in_train": flag, "n": len(gr), **{f"n_{c}": int((gr[LABEL] == c).sum()) for c in CLASSES},
               "macro_f1": f1_score(gr[LABEL], gr["pred"], average="macro", zero_division=0),
               **{f"f1_{c}": x for c, x in zip(CLASSES, f1s)}})
print("blog validation split by twin status (no twin = closer to test conditions):")
display(pd.DataFrame(tw).set_index("has_twin_in_train").round(3))

no_twin = ~vb["has_twin_in_train"].values
if no_twin.sum() > 0:
    fig, ax = plt.subplots(figsize=(6, 5))
    cm_plot(vb[LABEL].values[no_twin], vb["pred"].values[no_twin], f"blog, no twin in train (n={no_twin.sum():,})", ax)
    plt.tight_layout(); plt.show()

### Blog: confidence and the role of surface markers
Probability histograms show whether `fusion` texts are confidently wrong or close to the decision cut-off. The marker table shows accuracy for texts with and without typical markers: `urlLink` / `nbsp` (original 2004 corpus), and curly apostrophe / em dash / pictograph emoji (modern LLM style).

In [ ]:
bp = pd.DataFrame(val_proba[bm], columns=model.classes_).assign(true=yb)
plt.figure(figsize=(12, 3.8))
for i, c in enumerate(CLASSES):
    plt.subplot(1, 3, i + 1)
    sns.histplot(data=bp, x=c, hue="true", hue_order=CLASSES, bins=25, element="step", stat="density", common_norm=False)
    plt.title(f"P({c}) on blog val"); plt.xlim(0, 1)
plt.tight_layout(); plt.show()

PICTO = "[" + chr(0x1F300) + "-" + chr(0x1FAFF) + "]"
BLOG_MARKERS = {"urlLink / nbsp": r"urlLink|nbsp", "curly apostrophe": "’", "em dash": "—",
                "pictograph emoji": PICTO, "lowercase i": r"(?<!\w)i(?!\w)",
                "contraction w/o apostrophe": r"(?i)\b(dont|cant|im|didnt|thats)\b"}
mrows = []
for name, pat in BLOG_MARKERS.items():
    has = vb[TEXT].str.contains(pat, regex=True).values
    for flag in (True, False):
        gr = vb[has == flag]
        if len(gr):
            mrows.append({"marker": name, "present": flag, "n": len(gr),
                          **{f"share_{c}": (gr[LABEL] == c).mean() for c in CLASSES},
                          "accuracy": gr["correct"].mean(), "macro_f1": f1_score(gr[LABEL], gr["pred"], average="macro", zero_division=0)})
display(pd.DataFrame(mrows).set_index(["marker", "present"]).style.background_gradient(subset=["accuracy", "macro_f1"],
        cmap="RdYlGn", vmin=0.4, vmax=1).format(precision=3))

vb_len = pd.qcut(vb["n_chars"], 5, duplicates="drop")
print("blog macro-F1 by text length:")
display(pd.DataFrame({str(k): {"n": len(gr), "macro_f1": f1_score(gr[LABEL], gr["pred"], average="macro", zero_division=0),
                               "f1_fusion": f1_score(gr[LABEL], gr["pred"], labels=["fusion"], average=None, zero_division=0)[0]}
                      for k, gr in vb.groupby(vb_len, observed=True)}).T.round(3))

## By text length
Long texts are where a fusion insertion can be diluted, and very short texts give the model little to go on.

In [ ]:
v["length_bucket"] = pd.qcut(v["n_chars"], 5, duplicates="drop")
len_rows = []
for b, gr in v.groupby("length_bucket", observed=True):
    f1s = f1_score(gr[LABEL], gr["pred"], labels=CLASSES, average=None, zero_division=0)
    len_rows.append({"chars": str(b), "n": len(gr), "macro_f1": f1_score(gr[LABEL], gr["pred"], average="macro"),
                     **{f"f1_{c}": x for c, x in zip(CLASSES, f1s)}})
display(pd.DataFrame(len_rows).style.background_gradient(subset=["macro_f1", "f1_human", "f1_ai", "f1_fusion"],
                                                         cmap="RdYlGn", vmin=0.3, vmax=1).format(precision=3))

## Error examples

In [ ]:
err = v[~v["correct"]]
print("most common confusions:")
display(err.groupby([LABEL, "pred"]).size().sort_values(ascending=False).rename("n").to_frame())
print("most errors by domain (share of that domain's validation rows):")
display((err.groupby([LANG, DOMAIN]).size() / v.groupby([LANG, DOMAIN]).size()).dropna().sort_values(ascending=False)
        .round(3).rename("error_rate").to_frame())
for (t, pr), gr in err.groupby([LABEL, "pred"]):
    print(f"\n--- true={t} -> pred={pr} ({len(gr)})")
    for _, row in gr.sample(min(2, len(gr)), random_state=RANDOM_STATE).iterrows():
        print(f"   [{row[LANG]}|{row[DOMAIN]}]", repr(row[TEXT][:220]))
print("\n--- Gutenberg errors")
for _, row in err[err[DOMAIN] == GUT].head(6).iterrows():
    print(f"   true={row[LABEL]} pred={row['pred']} [{row[LANG]}]", repr(row[TEXT][:220]))
print("\n--- en-EN blog errors (no twin in train)")
eb = err[focus_mask(err) & ~err["has_twin_in_train"]]
for (t, pr), gr in eb.groupby([LABEL, "pred"]):
    print(f"  true={t} -> pred={pr} ({len(gr)})")
    for _, row in gr.sample(min(2, len(gr)), random_state=RANDOM_STATE).iterrows():
        print("     ", repr(row[TEXT][:260]))

## Predict test & save submission
Saves `/kaggle/working/submission.csv` (format `id,label`, in `sample_submission.csv` order) and shows a download link. On Kaggle, it also appears under *Output*, where it can be submitted to the competition.

In [ ]:
with timer("predict test"):
    test_pred = model.predict(X_test)

sub = sample_sub[["id"]].merge(pd.DataFrame({"id": test["id"].values, "label": test_pred}), on="id", how="left")
assert sub["label"].notna().all() and len(sub) == len(sample_sub) and set(sub["label"]) <= set(CLASSES)

OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
SUB_PATH = OUT_DIR / "submission.csv"
sub.to_csv(SUB_PATH, index=False)
print(f"saved {SUB_PATH.resolve()} {sub.shape} | validation macro-F1 {VAL_MACRO_F1:.4f}")
display(sub.head())
display(pd.concat({"train share": train[LABEL].value_counts(normalize=True),
                   "test pred share": sub["label"].value_counts(normalize=True),
                   "test pred share (gutenberg)": sub.loc[sub["id"].isin(test.loc[test[DOMAIN] == GUT, "id"]), "label"].value_counts(normalize=True),
                   "test pred share (en-EN blog)": sub.loc[sub["id"].isin(test.loc[focus_mask(test), "id"]), "label"].value_counts(normalize=True)},
                  axis=1).reindex(CLASSES).round(3))

os.chdir(OUT_DIR)            # FileLink paths are relative to the working directory
display(FileLink(SUB_PATH.name))